# Round 3 — Current Trading Strategies

This notebook documents the strategies in `ROUND_3/trader_final.py`.

**Backtest score**: ~$137,882 (historical 3-day); ~$7,855 on the actual submission replay  
**Products**: VELVETFRUIT_EXTRACT, HYDROGEL_PACK, VEV_4000–VEV_6500

---
## Product Overview

| Product | Type | Fair Value | Spread | Strategy |
|---------|------|-----------|--------|----------|
| VELVETFRUIT_EXTRACT | Spot underlying | ~5255 | 2–4 ticks | Mean-revert MM + insider tilt |
| HYDROGEL_PACK | Independent spot | ~9979–9990 | 12–18 ticks | Adaptive-fair MM |
| VEV_4000, VEV_4500 | Deep ITM calls | intrinsic | 15–25 ticks | Flash-arb |
| VEV_5000 | ATM call | intrinsic + ~13 | 4–6 ticks | Extrinsic-anchor MM |
| VEV_5100, VEV_5200 | Near-ATM calls | BS w/ smile | 2–4 ticks | BS smile MM |
| VEV_5300 | OTM call | intrinsic + ~47 | 2 ticks | Static-extrinsic MM + vev_dir tilt |
| VEV_5400 | OTM call | intrinsic + ~16 | 1–2 ticks | Static-extrinsic MM |
| VEV_5500 | Wing call | intrinsic + ~6 | 1 tick | Wing scalp |
| VEV_6000, VEV_6500 | Far wing | ~0.5 | 1 tick | **Skipped** (worthless) |

---
## Shared Infrastructure

In [11]:
import math
from dataclasses import dataclass
from statistics import NormalDist
from typing import Dict, Optional, Tuple

NORMAL = NormalDist()
TTE = 1.0          # normalized time-to-expiry (absorbed by smoothed IV)
IV_LO, IV_HI = 1e-4, 0.08


# ── Black-Scholes (statistics.NormalDist, no scipy) ──────────────────────────

def bs_call(S, K, T, sigma):
    if T <= 0 or sigma <= 0:
        return max(S - K, 0.0)
    sq = math.sqrt(T)
    d1 = (math.log(S / K) + 0.5 * sigma**2 * T) / (sigma * sq)
    d2 = d1 - sigma * sq
    return S * NORMAL.cdf(d1) - K * NORMAL.cdf(d2)

def bs_vega(S, K, T, sigma):
    if T <= 0 or sigma <= 0:
        return 0.0
    sq = math.sqrt(T)
    d1 = (math.log(S / K) + 0.5 * sigma**2 * T) / (sigma * sq)
    return S * NORMAL.pdf(d1) * sq

def bs_delta(S, K, T, sigma):
    if T <= 0 or sigma <= 0:
        return 1.0 if S > K else 0.0
    sq = math.sqrt(T)
    d1 = (math.log(S / K) + 0.5 * sigma**2 * T) / (sigma * sq)
    return NORMAL.cdf(d1)

def implied_vol(S, K, T, market_price, sigma_init=0.03, iters=30):
    """Newton-Raphson IV solver, clamped to [IV_LO, IV_HI]."""
    if market_price <= max(S - K, 0.0) + 1e-6:
        return IV_LO
    sigma = sigma_init
    for _ in range(iters):
        diff = bs_call(S, K, T, sigma) - market_price
        if abs(diff) < 1e-6:
            break
        v = bs_vega(S, K, T, sigma)
        if v < 1e-8:
            break
        sigma = max(IV_LO, min(IV_HI, sigma - diff / v))
    return sigma

# Quick sanity checks
S, K, T, iv = 5255, 5200, 1.0, 0.031
price = bs_call(S, K, T, iv)
iv_back = implied_vol(S, K, T, price)
print(f"BS call(S={S}, K={K}, T={T}, σ={iv:.4f}) = {price:.2f}")
print(f"IV round-trip: {iv:.4f} → {iv_back:.4f}  (diff = {abs(iv - iv_back):.2e})")
print(f"Delta = {bs_delta(S, K, T, iv):.3f}")

BS call(S=5255, K=5200, T=1.0, σ=0.0310) = 95.83
IV round-trip: 0.0310 → 0.0310  (diff = 7.29e-12)
Delta = 0.639


### Wall-Mid (hedgehogs method)

Rather than using `(best_bid + best_ask) / 2`, we use the **wall mid**: average of the *highest-volume* level on the bid and the *highest-volume* level on the ask.  
Designated MM bots park their liquidity at these walls, making it more stable than the bare top-of-book.

In [12]:
@dataclass
class Quote:
    bid: Optional[int] = None
    bid_vol: int = 0
    ask: Optional[int] = None
    ask_vol: int = 0
    bid_wall: Optional[int] = None   # price level with largest bid volume
    ask_wall: Optional[int] = None   # price level with largest ask volume

    @property
    def mid(self):
        if self.bid is None or self.ask is None: return None
        return (self.bid + self.ask) / 2.0

    @property
    def wall_mid(self):
        bw = self.bid_wall if self.bid_wall is not None else self.bid
        aw = self.ask_wall if self.ask_wall is not None else self.ask
        if bw is None or aw is None: return None
        return (bw + aw) / 2.0

    @property
    def spread(self):
        if self.bid is None or self.ask is None: return None
        return self.ask - self.bid


def quote_from(order_depth) -> Quote:
    q = Quote()
    if order_depth is None:
        return q
    if order_depth.buy_orders:
        q.bid = max(order_depth.buy_orders)
        q.bid_vol = order_depth.buy_orders[q.bid]
        q.bid_wall = max(order_depth.buy_orders, key=lambda p: order_depth.buy_orders[p])
    if order_depth.sell_orders:
        q.ask = min(order_depth.sell_orders)
        q.ask_vol = order_depth.sell_orders[q.ask]
        q.ask_wall = min(order_depth.sell_orders, key=lambda p: -order_depth.sell_orders[p])
    return q


def update_ema(emas: Dict, key: str, value: float, window: int) -> float:
    if key not in emas:
        emas[key] = value
        return value
    alpha = 2.0 / (window + 1.0)
    emas[key] = alpha * value + (1.0 - alpha) * emas[key]
    return emas[key]

print("Quote dataclass + update_ema defined.")

Quote dataclass + update_ema defined.


---
## 1. HYDROGEL_PACK — Adaptive-Fair MM

**Why not hardcode 10000?**  
The historical data shows true mean ≈ 9990, but the actual submission data shows it ≈ 9979.  
Hardcoding 10000 caused us to buy aggressively and sit at the +200 position limit with a −$11K drawdown.

**Fix**: rolling EMA fair = `0.4 * EMA(30) + 0.6 * EMA(200)` over wall_mid.  
This adapts to whatever regime we're in without overfitting.

| Param | Value | Why |
|-------|-------|-----|
| Fast EMA window | 30 | Reacts to short-term drift |
| Slow EMA window | 200 | Anchors to longer-run mean |
| Take edge | 7 ticks | > half the 12-tick typical spread |
| Quote edge | 4 ticks | Inside the spread, > 0 edge |
| Min spread to quote | 10 ticks | Don't quote into a tight book |

In [13]:
def trade_hydro_fair(wall_mid_series):
    """Simulate the rolling EMA fair on a price series."""
    emas = {}
    fairs = []
    for w in wall_mid_series:
        fast = update_ema(emas, 'fast', w, 30)
        slow = update_ema(emas, 'slow', w, 200)
        fairs.append(0.4 * fast + 0.6 * slow)
    return fairs

# Simulate: what if HYDRO stayed at 9979 the whole time?
import random
random.seed(42)
series = [9979 + random.gauss(0, 8) for _ in range(500)]

fairs = trade_hydro_fair(series)

# What fraction of ticks does the rolling fair correctly NOT trigger a buy
# (i.e., fair is below the ask)?
ask = 9990  # hypothetical constant ask at 10000 regime
static_buys = sum(1 for _ in series if ask < 10000 - 7.0)
ema_buys = sum(1 for f in fairs if ask < f - 7.0)
print(f"Static fair=10000: takes every tick where ask<9993 → always buys")
print(f"EMA fair adapts: after warmup, fair ≈ 9979, very few spurious buys")
print(f"Final EMA fair: {fairs[-1]:.1f}  (true mean: 9979)")

Static fair=10000: takes every tick where ask<9993 → always buys
EMA fair adapts: after warmup, fair ≈ 9979, very few spurious buys
Final EMA fair: 9978.4  (true mean: 9979)


---
## 2. VELVETFRUIT_EXTRACT — Mean-Revert MM + Insider Tilt

### 2a. Base MM

Fair value = `0.7 * EMA_fast(wall_mid, 15) + 0.3 * 5255`  
- Heavier weight on current mid (0.7) → less anchor drag  
- 5255 = midpoint between historical mean (5250) and submission mean (5262)
- Position skew: `fair -= 0.015 * position` to mean-revert inventory
- **Saturated MM**: post `POS_LIMIT` size on both sides (no cross-book guard) — matches the winning bot behavior

### 2b. Accumulator (Insider) Signal

From `trader_identification.ipynb`: the **Accumulator** is an insider bot.
- Trades of size ≥ 9 in VELVET are 99% buy-side
- Predicts a +2 tick mid move with **83% hit rate** over the next 500 ticks

When we see a ≥9 lot VELVET market trade:  
1. Shift fair up by +1.5 ticks for 1500 ticks  
2. Lift the ask directly (up to 25 lots)

**Note**: `vev_dir` (option flow signal) is NOT used here. Testing showed it hurts VELVET by −$60K.

In [14]:
def simulate_velvet_fair(wall_mids, positions, v_active_flags):
    """Show how fair evolves with and without insider tilt."""
    emas = {}
    fairs_base, fairs_tilt = [], []
    for wall, pos, active in zip(wall_mids, positions, v_active_flags):
        fast = update_ema(emas, 'vfast', wall, 15)
        fair_base = 0.7 * fast + 0.3 * 5255.0 - 0.015 * pos
        fair_tilt = fair_base + (1.5 if active else 0.0)
        fairs_base.append(fair_base)
        fairs_tilt.append(fair_tilt)
    return fairs_base, fairs_tilt

# Example: VELVET sits at 5255, then Accumulator fires at tick 100
walls = [5255 + random.gauss(0, 3) for _ in range(200)]
pos = [0] * 200
v_active = [False] * 100 + [True] * 50 + [False] * 50  # signal active 100-150
fb, ft = simulate_velvet_fair(walls, pos, v_active)
print(f"Tick 90  — base fair: {fb[90]:.2f}, tilted: {ft[90]:.2f}")
print(f"Tick 110 — base fair: {fb[110]:.2f}, tilted: {ft[110]:.2f}  (+{ft[110]-fb[110]:.2f} tilt)")
print(f"Tick 160 — base fair: {fb[160]:.2f}, tilted: {ft[160]:.2f}  (signal off)")

Tick 90  — base fair: 5254.92, tilted: 5254.92
Tick 110 — base fair: 5254.81, tilted: 5256.31  (+1.50 tilt)
Tick 160 — base fair: 5255.21, tilted: 5255.21  (signal off)


---
## 3. VEV_4000 / VEV_4500 — Flash-Arb (Deep ITM)

### The Flash Bug

Deep ITM options (K=4000, 4500) should have near-zero extrinsic (they're 1000+ ticks in the money). But we observed ~520 times per day where their mid **dislocates** by ±5–7 ticks away from intrinsic.

**Cause**: the Wing Seller bot periodically dumps a basket of OTM calls. The option MM bot adjusts all quotes simultaneously, creating a brief window where deep-ITM options are mispriced.

**Detection**: `surface_shift = avg(opt_mid − max(S−K, 0))` across VEV_4000 and VEV_4500.  
- `flash_down = surface_shift <= −4` → buy (options too cheap)
- `flash_up = surface_shift >= +4` → sell (options too expensive)
- Also take on `actual_cheap = ask <= fair − 2` regardless of shift

### Wing Signal Boost
If wing products (VEV_5300–6500) show any market trades this tick, the flash-arb size increases by 3.

In [15]:
# Simulate flash detection logic
def detect_flash(opt_mid_4000, opt_mid_4500, S):
    shifts = []
    for opt_mid, K in [(opt_mid_4000, 4000), (opt_mid_4500, 4500)]:
        if opt_mid is not None:
            shifts.append(opt_mid - max(S - K, 0.0))
    surface_shift = sum(shifts) / len(shifts) if shifts else None
    flash_down = surface_shift is not None and surface_shift <= -4.0
    flash_up = surface_shift is not None and surface_shift >= 4.0
    return surface_shift, flash_down, flash_up

# Normal tick: extrinsic ~ 0
S = 5255.0
shift, fd, fu = detect_flash(1255.0, 755.0, S)  # intrinsic = 5255-4000=1255, 5255-4500=755
print(f"Normal:     shift={shift:.1f}, flash_down={fd}, flash_up={fu}")

# Flash-down: Wing Seller dumps, surface drops
shift, fd, fu = detect_flash(1249.0, 749.0, S)  # -6 below intrinsic
print(f"Flash-down: shift={shift:.1f}, flash_down={fd}, flash_up={fu}")

# Flash-up
shift, fd, fu = detect_flash(1261.0, 761.0, S)  # +6 above intrinsic
print(f"Flash-up:   shift={shift:.1f}, flash_down={fd}, flash_up={fu}")

Normal:     shift=0.0, flash_down=False, flash_up=False
Flash-down: shift=-6.0, flash_down=True, flash_up=False
Flash-up:   shift=6.0, flash_down=False, flash_up=True


---
## 4. VEV_5000 — ATM Extrinsic-Anchor MM

Near-ATM options have delta ≈ 0.5, so they're exposed to VELVET drift. Rather than full BS, we use a simpler approach:

- Observe extrinsic = `mid − max(S − 5000, 0)`
- EMA-smooth it (window 80) and blend 50/50 with anchor=13
- Cap drift to ±6 around anchor to protect against flash ticks

This "half-static" approach avoids the gamma bleeding of pure static fair.

In [16]:
def trade_5000_fair(S_series, opt_mid_series, anchor=13.0):
    emas = {}
    fairs = []
    for S, opt_mid in zip(S_series, opt_mid_series):
        intrinsic = max(S - 5000, 0.0)
        obs_ext = opt_mid - intrinsic
        ext_ema = update_ema(emas, 'ext', obs_ext, 80)
        ext_blend = max(anchor - 6, min(anchor + 6, 0.5 * ext_ema + 0.5 * anchor))
        fairs.append(intrinsic + ext_blend)
    return fairs

# S bounces around 5255, extrinsic observed around 13
S_vals = [5255 + random.gauss(0, 5) for _ in range(100)]
opt_mid_vals = [max(s - 5000, 0) + 13 + random.gauss(0, 1) for s in S_vals]
fairs = trade_5000_fair(S_vals, opt_mid_vals)
print(f"Fair range: [{min(fairs):.1f}, {max(fairs):.1f}]")
print(f"Final fair: {fairs[-1]:.2f}  (expected ~{max(S_vals[-1]-5000,0)+13:.2f})")

Fair range: [257.0, 286.6]
Final fair: 267.57  (expected ~267.55)


---
## 5. VEV_5100 / VEV_5200 — Black-Scholes with Smile Interpolation

### Smile Shape

Calibrated from historical data. The smile is parametrized as IV offsets keyed by **moneyness = (K − S) / 100**:

| Moneyness | IV offset |
|-----------|----------|
| −2.5 | −0.0006 |
| −1.5 | −0.0012 |
| −0.5 | −0.0003 |
| +0.5 | 0.0 |
| +1.5 | −0.0019 |
| +2.5 | +0.0005 |
| +7.5 | −0.0040 |
| +12.5 | −0.0060 |

### Smile Level

We don't fix the IV level. Instead, we observe IV across all quoted strikes (5000–5500), spread-weight them, and EMA-smooth the result. This gives a **smile level** that drifts slowly with the market.

`smile_level = EMA(spread-weighted avg of (iv_smooth − smile_offset))`

In [17]:
SMILE_KNOTS = (
    (-2.5, -0.0006), (-1.5, -0.0012), (-0.5, -0.0003),
    (0.5, 0.0), (1.5, -0.0019), (2.5, 0.0005),
    (7.5, -0.0040), (12.5, -0.0060),
)

def smile_offset(moneyness):
    if moneyness <= SMILE_KNOTS[0][0]: return SMILE_KNOTS[0][1]
    if moneyness >= SMILE_KNOTS[-1][0]: return SMILE_KNOTS[-1][1]
    for (x0, y0), (x1, y1) in zip(SMILE_KNOTS, SMILE_KNOTS[1:]):
        if x0 <= moneyness <= x1:
            return y0 + (y1 - y0) * (moneyness - x0) / (x1 - x0)
    return 0.0

# Show the smile shape
S_ref = 5255.0
strikes = [4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500]
smile_lv = 0.031  # typical smile level

print(f"{'Strike':>7}  {'Moneyness':>9}  {'IV offset':>9}  {'IV':>6}  {'BS Price':>9}")
for K in strikes:
    m = (K - S_ref) / 100.0
    offset = smile_offset(m)
    iv = max(IV_LO, min(IV_HI, smile_lv + offset))
    price = bs_call(S_ref, K, TTE, iv)
    print(f"  {K:>5}  {m:>9.2f}  {offset:>+9.4f}  {iv:.4f}  {price:>9.2f}")

 Strike  Moneyness  IV offset      IV   BS Price
   4000     -12.55    -0.0006  0.0304    1255.00
   4500      -7.55    -0.0006  0.0304     755.00
   5000      -2.55    -0.0006  0.0304     258.32
   5100      -1.55    -0.0012  0.0298     167.78
   5200      -0.55    -0.0003  0.0307      95.16
   5300       0.45    -0.0000  0.0310      45.19
   5400       1.45    -0.0018  0.0292      14.71
   5500       2.45    +0.0004  0.0314       5.51
   6000       7.45    -0.0040  0.0270       0.00
   6500      12.45    -0.0060  0.0250       0.00


### Why raw mid (not wall_mid) for options?

The smile fit was calibrated against raw mid prices. Switching to wall_mid subtly shifts all implied vols and bleeds PnL because the BS model is calibrated differently than what the market prices. VELVET's MM benefits from wall_mid; the options strategies keep raw mid.

---
## 6. VEV_5300 — Static Extrinsic + vev_dir Tilt

**Strategy**: simple MM around `fair = max(S − 5300, 0) + 47`

The 47-tick extrinsic is empirically stable from the submission log.  
The OTM option has low delta (~0.05–0.15) so static fair holds up even as VELVET drifts.

### vev_dir signal

From Discord tip: when **both** VEV_5200 and VEV_5300 spreads are ≤ 2 simultaneously, trades in those options carry directional information (the informed bot hedges into the tight surface).

- Score market trades by side (ask-hit = bullish, bid-hit = bearish)
- If score > 0 → `vev_dir = +1` for 1200 ticks
- If score < 0 → `vev_dir = -1` for 1200 ticks
- Fair shifts by `2 * vev_dir`

**Note**: vev_dir helps VEV_5300 but NOT VELVET directly (tested: −$60K on VELVET).

In [18]:
def compute_vev_dir(trades_5200, trades_5300, q5200_mid, q5300_mid, q5200_bid, q5200_ask, q5300_bid, q5300_ask):
    """Compute directional score from option flow."""
    score = 0
    for trades, q_mid, q_bid, q_ask in [
        (trades_5200, q5200_mid, q5200_bid, q5200_ask),
        (trades_5300, q5300_mid, q5300_bid, q5300_ask),
    ]:
        if q_mid is None:
            continue
        for price, qty in trades:
            n = abs(qty)
            if q_ask is not None and price >= q_ask:
                score += n  # lifting the ask = bullish
            elif q_bid is not None and price <= q_bid:
                score -= n  # hitting the bid = bearish
            elif price > q_mid:
                score += n
            elif price < q_mid:
                score -= n
    return score

# Example: buyers lifting the ask on both strikes
trades = [(5253, 5), (5253, 3)]  # 5300 ask-side trades (ask ~ 5253)
score = compute_vev_dir([], trades, 5251.5, None, 5250, 5253, None, None)
print(f"Bullish option flow: score = {score}  → vev_dir = {1 if score > 0 else (-1 if score < 0 else 0)}")

Bullish option flow: score = 0  → vev_dir = 0


---
## 7. VEV_5400 — Static Extrinsic = 16, Uncapped Take

**Strategy**: MM around `fair = max(S − 5400, 0) + 16`

Key difference from 5300: **take size is uncapped** — we take the full available top-of-book volume.

Why? The VEV_5400 book typically has only 1–2 lot depth. Capping at 6 left $300 on the table vs trader_strats's uncapped approach.

---
## 8. VEV_5500 — Wing Scalp

**Strategy**: scalp around `fair = max(S − 5500, 0) + 6` with EMA-smoothed extrinsic, capped at anchor ± 4.

Very small position sizes (4 lots) and conservative edges (3 ticks to take). Only tries to exit inventory at fair+1 if long, or enter at fair−1 if short.

---
## 9. VEV_6000, VEV_6500 — Skipped

These options are essentially worthless (mid ≈ 0.5, floored at 0). The spread is 1 tick on a 0.5 mid, so any MM is quoting on the wrong side of 0. No strategy.

---
## Strategy Interaction Summary

In [19]:
import textwrap

summary = [
    ("HYDROGEL_PACK",       "Adaptive EMA",     "None",          "$1,441 on submission"),
    ("VELVETFRUIT_EXTRACT", "EMA MM",           "Accumulator",   "Largest P&L driver"),
    ("VEV_4000",            "Flash-arb",        "Wing signal",   "~$3K/day backtest"),
    ("VEV_4500",            "Flash-arb",        "Wing signal",   "~$2K/day backtest"),
    ("VEV_5000",            "Extrinsic anchor", "None",          "Steady small edge"),
    ("VEV_5100",            "BS smile",         "v_active delta","Near-ATM alpha"),
    ("VEV_5200",            "BS smile",         "v_active delta","Near-ATM alpha"),
    ("VEV_5300",            "Static ext=47",    "vev_dir",       "~$4K/day backtest"),
    ("VEV_5400",            "Static ext=16",    "None",          "~$4K/day backtest"),
    ("VEV_5500",            "Wing scalp",       "None",          "Small / defensive"),
]

print(f"{'Product':<25} {'Base Strategy':<20} {'Signal Used':<20} {'Notes'}")
print("-" * 90)
for row in summary:
    print(f"{row[0]:<25} {row[1]:<20} {row[2]:<20} {row[3]}")

Product                   Base Strategy        Signal Used          Notes
------------------------------------------------------------------------------------------
HYDROGEL_PACK             Adaptive EMA         None                 $1,441 on submission
VELVETFRUIT_EXTRACT       EMA MM               Accumulator          Largest P&L driver
VEV_4000                  Flash-arb            Wing signal          ~$3K/day backtest
VEV_4500                  Flash-arb            Wing signal          ~$2K/day backtest
VEV_5000                  Extrinsic anchor     None                 Steady small edge
VEV_5100                  BS smile             v_active delta       Near-ATM alpha
VEV_5200                  BS smile             v_active delta       Near-ATM alpha
VEV_5300                  Static ext=47        vev_dir              ~$4K/day backtest
VEV_5400                  Static ext=16        None                 ~$4K/day backtest
VEV_5500                  Wing scalp           None            

---
## Backtest Results History

| Version | Description | Backtest |
|---------|-------------|----------|
| trader_overfit | Static fair=10k HYDRO, BS options | $118K |
| trader_strats | Voucher-gated insider, VEV_5300/5400 MM | $92K |
| trader_smile_surface | BS smile for 5100/5200, flash-arb ITM | $187K |
| trader_final (v1) | Best-of-each combined, vev_dir in VELVET (bug) | $148K |
| trader_final (v2) | Fixed: vev_dir only in 5300, wall_mid in VELVET | $227K |
| trader_final (v3) | Fixed: rolling EMA HYDRO fair (anti-overfit) | $137K |

The $137K backtest is lower than $227K because we removed the HYDRO static-fair overfit.  
On actual submission data, the rolling EMA version scores $7,855 vs $3,427 for the static version.